In [1]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import os

In [2]:
spark = (SparkSession.builder
         .appName("MySpark")
         .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
         .master("local[*]")
         .getOrCreate())

In [3]:
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_password = os.getenv("POSTGRES_PASSWORD")
pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = os.getenv("POSTGRES_PORT", 5432)
pg_db = "postgres"

jdbc_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"

base_reader = (spark.read
               .format("jdbc")
               .option("url", jdbc_url)
               .option("driver", "org.postgresql.Driver")
               .option("user", pg_user)
               .option("password", pg_password))


In [4]:
csv_path = "/data/*.csv"
base_df = spark.read.csv(csv_path, header=True, inferSchema=True, quote='"', multiLine=True, escape='"', ignoreLeadingWhiteSpace=True, ignoreTrailingWhiteSpace=True, sep=',')

w_main = Window.orderBy(lit(1))
df = base_df.withColumn("global_id", row_number().over(w_main))

In [5]:
def write_to_table(df, table):
    (df.write
       .format("jdbc")
       .option("url", jdbc_url)
       .option("driver", "org.postgresql.Driver")
       .option("user", pg_user)
       .option("password", pg_password)
       .option("dbtable", table)
       .mode("append")
       .save())

In [6]:
countries = df.select(col("customer_country").alias("name")).union(df.select("seller_country")) \
    .union(df.select("store_country")).union(df.select("supplier_country")) \
    .distinct().na.drop()
dim_countries = countries.withColumn("id", row_number().over(Window.orderBy("name")))

pet_cats = df.select(col("pet_category").alias("category")).distinct().na.drop()
dim_pet_categories = pet_cats.withColumn("id", row_number().over(Window.orderBy("category")))

write_to_table(dim_countries, "dim_countries")
write_to_table(dim_pet_categories, "dim_pet_categories")

In [7]:
def join_country(target_df, country_col):
    return (target_df.join(dim_countries.withColumnRenamed('id', 'country_id'), target_df[country_col] == dim_countries["name"], "left")
            .drop("name", country_col))

In [8]:
customers = df.select(
    col("global_id").alias("id"), "customer_first_name", "customer_last_name",
    "customer_age", "customer_email", "customer_country", "customer_postal_code",
    "customer_pet_type", "customer_pet_name", "customer_pet_breed", "pet_category"
)

customers = join_country(customers, "customer_country")

dim_customers = (customers.join(
    dim_pet_categories.withColumnRenamed("id", "pet_category_id"),
    customers["pet_category"] == dim_pet_categories["category"], "left")
                 .drop("category", "pet_category"))

dim_suppliers = join_country(df.select(
    col("global_id").alias("id"), "supplier_name", "supplier_contact",
    "supplier_email", "supplier_phone", "supplier_address", "supplier_city", "supplier_country"
), "supplier_country")

dim_stores = join_country(df.select(
    col("global_id").alias("id"), "store_name", "store_location", "store_city",
    "store_state", "store_country", "store_phone", "store_email"
), "store_country")

dim_sellers = join_country(df.select(
    col("global_id").alias("id"), "seller_first_name", "seller_last_name",
    "seller_email", "seller_country", "seller_postal_code",
    col("global_id").alias("seller_store_id")
), "seller_country")

dim_products = df.select(
    col("global_id").alias("id"), "product_name", "product_category", "product_price",
    "product_quantity", "product_weight", "product_color", "product_size",
    "product_brand", "product_material", "product_description", "product_rating",
    "product_reviews", "product_release_date", "product_expiry_date",
    col("global_id").alias("product_supplier_id")
)

In [9]:
fact_sales = df.select(
    col("global_id").alias("id"),
    "sale_date",
    "sale_quantity",
    "sale_total_price",
    col("global_id").alias("sale_customer_id"),
    col("global_id").alias("sale_seller_id"),
    col("global_id").alias("sale_product_id"),
    col("global_id").alias("sale_store_id"),
    col("global_id").alias("sale_supplier_id")
)

In [10]:
df = df.drop('id').withColumnRenamed('global_id', 'id')

In [11]:
write_to_table(dim_suppliers, "dim_suppliers")
write_to_table(dim_stores, "dim_stores")
write_to_table(dim_sellers, "dim_sellers")
write_to_table(dim_customers, "dim_customers")
write_to_table(dim_products, "dim_products")
write_to_table(fact_sales, "fact_sales")

write_to_table(df, 'mock_data')